# 분포 파악 차트 — 히스토그램 · KDE · 박스플롯


**직접 풀어보는 12문제:** 정답 코드는 넣지 않았습니다. 준비 코드만 제공하며 각 문제의 빈 셀에 풀이를 작성합니다. 점검 숫자는 아래 가상 데이터에 대한 값입니다. 이 노트북의 준비 데이터에만 점검 숫자를 적용하세요.

각 노트북은 독립적으로 사용할 수 있습니다. 위에서 아래로 진행하고, 실험에는 복사본을 사용하세요. 개별 문제에서 만든 결과는 다른 이름에 저장해 원본 변수를 덮어쓰지 않는 것을 권합니다.

## 처음 배우는 분을 위한 안내

히스토그램은 숫자 구간별 개수, 박스플롯은 중앙값과 가운데 50% 범위, KDE는 매끄럽게 그린 분포 곡선입니다. KDE의 세로축은 개수가 아닌 밀도입니다. bins는 구간 수, IQR은 Q3−Q1입니다. revenue는 매출, delivery_days는 배송일수입니다.

**학습 순서:** 문제 1 ~ 3은 필수 기초, KDE·사분위수 비교가 포함된 4 ~ 12는 선택 도전입니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

그래프 창이 비어 있으면 데이터 개수와 결측 제거 결과를 먼저 확인하세요. 한글이 네모로 나오면 글꼴 설정 문제이며 계산 오류와 구분합니다.

## 실행 준비

Python 3와 `pandas`, `numpy`, `matplotlib`, `seaborn`이 필요합니다(확장 시각화 노트북은 `wordcloud`, `folium` 추가 필요). 없는 경우 별도 코드 셀에서 `%pip install pandas numpy matplotlib seaborn wordcloud folium`을 한 번 실행하세요. 이 노트북은 외부 CSV 없이 실습 데이터가 자동 생성됩니다.

Colab 또는 Jupyter에서 `.ipynb`를 열고 **준비 셀부터 순서대로** 실행하세요. 아래 코드는 한글 폰트 설정과 실습 데이터 생성만 수행합니다.

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 운영체제별 한글 폰트 및 마이너스 기호 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 학습용 가상 데이터 (외부 CSV 없이 바로 실행 가능)
online = pd.DataFrame({
    'order_id': range(101, 117),
    'revenue': [120, 150, 180, 200, 220, 240, 250, 260,
                280, 300, 320, 350, 380, 420, 450, 850],
    'delivery_days': [2, 2, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 6, 6, 7, 11]
})

offline = pd.DataFrame({
    'order_id': range(201, 217),
    'unit_price': [20, 22, 24, 25, 25, 26, 27, 28,
                   29, 30, 31, 32, 34, 35, 75, 90],
    'revenue': [180, 210, 230, 260, 290, 310, 330, 360,
                390, 410, 440, 480, 520, 550, 600, 920],
    'delivery_days': [1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 5, 8]
})

summary = pd.DataFrame({
    'customer_id': [f'C{i:02d}' for i in range(1, 15)],
    'membership_tier': ['Bronze', 'Bronze', 'Silver', 'Silver', 'Bronze', 'Silver', 'Gold',
                        'Silver', 'Gold', 'Gold', 'Silver', 'Bronze', 'Gold', 'Gold'],
    'avg_order_value': [40.0, 55.0, 60.0, 75.0, None, 80.0, 95.0,
                        110.0, 125.0, 140.0, 160.0, None, 210.0, 320.0],
    'total_spent': [180, 220, 250, 270, 290, 300, 310,
                    340, 360, 400, 450, 480, 1200, 1500]
})

sales = pd.DataFrame({
    'region': ['South'] * 10 + ['North'] * 10,
    'delivery_days': [1, 2, 2, 2, 3, 3, 3, 4, 4, 5,
                      3, 4, 4, 5, 5, 6, 6, 7, 8, 12],
    'revenue': [150, 180, 210, 230, 260, 290, 310, 340, 390, 450,
                200, 240, 280, 320, 350, 380, 420, 460, 520, 780]
})

## 문제 1. 온라인 주문 매출 히스토그램 기본 [필수 기초]

`online`의 `revenue` 열을 사용하여 `bins=5`, `edgecolor='white'`인 히스토그램을 그리세요. X축 이름은 `Revenue`, Y축 이름은 `Frequency`, 그래프 제목은 `Online Order Revenue Distribution`으로 지정하세요.

**점검 기준:** `online['revenue']`는 총 16건(최소 120, 최대 850)이며, 200 ~ 350 주변 저·중가 구간에 대부분의 주문이 몰려 있고 오른쪽 꼬리가 긴 분포가 나타납니다.

In [ ]:
# 문제 1 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 2. 결측치를 제외한 고객 평균 주문 금액 히스토그램 [필수 기초]

`summary`의 `avg_order_value` 열에서 결측치 개수와 비결측 유효 데이터 개수를 먼저 출력하고, 결측치를 제외한 값으로 `bins=6` 히스토그램을 그리세요. 그래프 제목은 `Average Order Value Distribution`으로 지정하세요.

**점검 기준:** 원본 14행 중 결측치가 2개(`C05`, `C12`), 유효 표본은 12개입니다. 결측을 제외한 평균은 약 `122.92`, 중앙값은 `102.5`입니다.

In [ ]:
# 문제 2 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 3. `bins` 값에 따른 배송 기간 히스토그램 비교 [필수 기초]

`sales`의 `delivery_days` 열을 사용해 `1행 2열(figsize=(10, 4))` 서브플롯에 첫 번째는 `bins=4`, 두 번째는 `bins=10`인 히스토그램을 나란히 그리고 두 그래프의 모양 차이를 적으세요.

**점검 기준:** `bins=4`에서는 전체가 3 ~ 4개 덩어리로 단순화되어 보이지만, `bins=10`에서는 2 ~ 6일 구간의 세부 빈도 차이와 12일 극단값의 분리된 위치가 더 뚜렷하게 드러납니다.

In [ ]:
# 문제 3 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 4. 온라인과 오프라인 주문 매출 KDE 비교 [선택 도전]

`online['revenue']`와 `offline['revenue']`를 하나의 그래프에 `sns.kdeplot(fill=True)`로 겹쳐 그리세요. 각각 `Online`, `Offline` 라벨(`label`)을 지정하고 범례(`legend`)를 표시한 뒤 어느 채널의 분포가 더 오른쪽(고액)에 위치하는지 해석하세요.

**점검 기준:** `Online` 중앙값은 `255.0`(평균 `298.12`), `Offline` 중앙값은 `375.0`(평균 `405.0`)으로 `Offline` 곡선이 전반적으로 오른쪽(높은 매출 구간)에 위치합니다.

In [ ]:
# 문제 4 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 5. 고객별 총 구매금액 KDE와 비대칭(치우침) 확인 [선택 도전]

`summary['total_spent']`를 `sns.kdeplot(fill=True)`로 시각화하고 그래프 제목을 `Total Spent Distribution`으로 지정하세요. 평균과 중앙값을 함께 출력해 분포가 어느 쪽으로 치우쳐 있는지 설명하세요.

**점검 기준:** 14명의 평균은 `482.14`, 중앙값은 `325.0`입니다. 1200·1500 고액 고객 때문에 오른쪽 꼬리가 길게 뻗은 양의 비대칭(오른쪽 치우침) 분포를 보이며 평균이 중앙값보다 크게 나타납니다.

In [ ]:
# 문제 5 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 6. 두 지역의 배송 기간 분포 KDE 비교 [선택 도전]

`sales`에서 `region`의 고유값을 확인한 뒤, `South`와 `North` 두 지역의 `delivery_days` 분포를 하나의 그래프에 KDE 곡선으로 비교하세요. 라벨과 범례를 함께 표시하세요.

**점검 기준:** `South`(10건)의 배송 기간 중앙값은 `3.0일`(평균 `2.9일`)이고, `North`(10건)의 중앙값은 `5.5일`(평균 `6.0일`)로 `North` 지역의 배송 기간 분포가 더 길고 넓게 퍼져 있습니다.

In [ ]:
# 문제 6 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 7. 오프라인 상품 단가 박스플롯과 잠재적 이상치 [선택 도전]

`offline['unit_price']`를 `plt.boxplot()`으로 시각화하세요. Y축 이름을 `Unit Price`, 제목을 `Offline Unit Price Box Plot`으로 지정하고 수염 바깥에 찍히는 점들을 확인하세요.

**점검 기준:** 중앙값 선은 `28.5`에 위치하며, 위쪽 수염 바깥에 고가 상품 단가인 `75`와 `90` 두 개의 점이 잠재적 이상치로 표시됩니다.

In [ ]:
# 문제 7 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 8. 사분위수(Q1·Q3)와 IQR 이상치 경계 직접 계산하기 [선택 도전]

`offline['unit_price']`의 `Q1(25%)`, `중앙값(50%)`, `Q3(75%)`, `IQR(Q3 - Q1)`, 하한 경계(`Q1 - 1.5*IQR`), 상한 경계(`Q3 + 1.5*IQR`)를 계산해 출력하고, 이 범위를 벗어나는 값을 필터링해 문제 7의 박스플롯과 일치하는지 확인하세요.

**점검 기준:** `Q1 = 25.0`, `중앙값 = 28.5`, `Q3 = 32.5`, `IQR = 7.5`이며, 하한 경계는 `13.75`, 상한 경계는 `43.75`입니다. 상한을 초과하는 값은 `[75, 90]` 2개입니다.

In [ ]:
# 문제 8 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 9. 고객별 총 구매금액 박스플롯과 고액 구매 고객 탐색 [선택 도전]

`summary['total_spent']`의 박스플롯을 그리고, `1.5 × IQR` 상한 경계를 초과하는 고액 구매 고객의 `customer_id`와 `total_spent`를 출력하세요.

**점검 기준:** `Q1 = 275.0`, `Q3 = 437.5`, `IQR = 162.5`, 상한 경계는 `681.25`이며, 이를 초과하는 고객은 `C13(1200)`, `C14(1500)` 2명입니다.

In [ ]:
# 문제 9 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 10. 온라인과 오프라인 배송 기간 그룹 박스플롯 비교 [선택 도전]

`online['delivery_days']`와 `offline['delivery_days']`를 하나의 박스플롯 차트에 `Online`, `Offline` 두 그룹으로 나란히 그리고 중앙값, 박스 높이(IQR), 이상치 유무를 비교하세요.

**점검 기준:** `Online` 중앙값은 `4.0일`(IQR `2.25`, 이상치 `11일`), `Offline` 중앙값은 `3.0일`(IQR `2.0`, 이상치 `8일`)로 온라인 배송 기간이 전반적으로 약 1일 더 깁니다.

In [ ]:
# 문제 10 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 11. Seaborn `histplot(hue=...)`과 `displot(kde=True)` 활용 [선택 도전]

`summary` 데이터에서 `sns.boxplot(data=summary, x='membership_tier', y='total_spent')`로 등급별 총 구매금액 분포를 비교하고, 이어서 `sales`의 `revenue`에 대해 `sns.displot(data=sales, x='revenue', hue='region', kde=True, bins=6)`을 실행해 히스토그램과 KDE를 동시에 확인하세요.

**점검 기준:** `Gold` 등급의 구매금액 분포가 `Bronze`·`Silver`보다 높게 형성되며, `displot(kde=True)`을 통해 막대 빈도와 부드러운 추세 곡선을 한 번에 비교할 수 있습니다.

In [ ]:
# 문제 11 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 12. 같은 매출 데이터를 세 가지 차트(1×3 Subplots)로 종합 비교 [선택 도전]

`online['revenue']`를 `1행 3열` 서브플롯(`figsize=(13, 4)`)에 ① 히스토그램(`bins=5`), ② KDE(`fill=True`), ③ 박스플롯으로 각각 그리고, 세 차트에서 가장 파악하기 쉬운 특징이 무엇인지 한 문장씩 정리하세요.

**점검 기준:** 히스토그램은 200 ~ 350 구간의 주문 건수 집중을, KDE는 오른쪽으로 긴 비대칭 꼬리 형태를, 박스플롯은 중앙값(`255.0`)과 상단 이상치(`850`)의 존재를 가장 명확하게 보여줍니다.

In [ ]:
# 문제 12 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 선택 도전: 다섯 개 금액의 중심과 꼬리 비교

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
import pandas as pd
extra_values = pd.Series([10,12,12,13,50],name='amount')
```

### 진행 순서

1. 평균과 중앙값을 출력합니다.
2. bins=4인 히스토그램을 그립니다. X축 Amount, Y축 Count, 제목 Small Amount Distribution을 붙입니다.
3. 50이 평균을 얼마나 끌어올렸는지 설명합니다.

**완료 기준:** 평균 19.4, 중앙값 12, 히스토그램의 빈도 합계 5. 실제 금액 단위는 이 가상 예제에서 지정하지 않음.

## 제출 전 확인

- 준비 셀과 풀이를 위에서 아래로 실행했나요?
- 그래프에 제목(`title`), 축 이름(`xlabel`, `ylabel`), 다중 계열인 경우 범례(`legend`)를 표시했나요?
- 모든 점검 기준에 대해 수치 근거와 시각적 특징을 함께 남겼나요?
- 결측값 처리 여부와 표본 크기(또는 구간/윈도우 설정)를 확인했나요?
- 원본 데이터프레임 변수를 덮어쓰지 않고 안전하게 실습했나요?